# YOLO11 Point — 10.000 train / 1.500 validation
Thứ tự góc: TL → TR → BR → BL. Bộ validation được giữ cố định.
Đã loại các ảnh trùng validation khỏi tập train trước khi xuất gói.

1. Chọn **Runtime → Change runtime type → GPU**.
2. Tự chép ZIP dữ liệu vào `MyDrive/nATime-YOLO11-Point-10000/datasets/`.
3. Lần đầu giữ `FULL_TRAIN=False` để chạy thử 200 train / 80 validation trong 2 epoch.
4. Sau khi chạy thử đạt, chọn `FULL_TRAIN=True`, `RESUME=False`, `RUN_ID=''` và chạy lại từ ô cấu hình để tạo lượt train 100 epoch.
5. Khi Colab ngắt: chọn đúng FULL_TRAIN, đặt RESUME=True và RUN_ID cũ. Chỉ khôi phục tiến độ đã checkpoint; epoch đang chạy dở có thể phải chạy lại.

Không chạy hai phiên vào cùng RUN_ID. Khởi tạo từ YOLO11n-Pose COCO gốc, không dùng model biển số cũ đã học validation. Mount Drive do bạn cấp quyền; notebook không gửi dữ liệu sang dịch vụ huấn luyện khác.


In [ ]:
%pip install -q ultralytics==8.4.183 Pillow==11.3.0 tensorboard onnx onnxruntime onnxslim
import torch,sys,subprocess
assert torch.cuda.is_available(), 'Hãy chọn GPU trong Runtime trước khi chạy'
print('GPU:',torch.cuda.get_device_name(0))
subprocess.run(['nvidia-smi'],check=True)


In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime,timezone
import uuid,re,hashlib,json,zipfile,os,shutil,random,yaml
drive.mount('/content/drive')
FULL_TRAIN=False
RESUME=False
RUN_ID=''  # Khi resume: dán RUN_ID đã in ở lần chạy trước.
IMGSZ=960
BATCH=4  # Có thể giảm khi thiếu VRAM; giữ IMGSZ nếu tiếp tục cùng run.
EPOCHS=100 if FULL_TRAIN else 2
DRIVE_ROOT=Path('/content/drive/MyDrive/nATime-YOLO11-Point-10000')
DATASET_ZIP=DRIVE_ROOT/'datasets/ANPR_YOLO11_Point_10000_train_1500_val.zip'
EXPECTED_ZIP_SHA256='814c356690ef3c283e3fe7d3e16724c3361824a2c3b0ffd8158dc458a2478683'
assert DATASET_ZIP.is_file(), f'Chép ZIP vào: {DATASET_ZIP}'
if RESUME:
    assert RUN_ID, 'Resume cần RUN_ID cũ'
elif not RUN_ID:
    RUN_ID=('full-' if FULL_TRAIN else 'smoke-')+datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-')+uuid.uuid4().hex[:6]
assert re.fullmatch(r'[A-Za-z0-9_-]{1,100}',RUN_ID)
DRIVE_RUN=DRIVE_ROOT/'runs'/RUN_ID
LOCAL_RUN=Path('/content/anpr-pose-runs')/RUN_ID
ROOT=Path('/content/anpr-point-10000-1500')
print('RUN_ID:',RUN_ID,'| GPU batch:',BATCH,'| epochs:',EPOCHS)


In [ ]:
with DATASET_ZIP.open('rb') as f:
    checksum=hashlib.file_digest(f,'sha256').hexdigest()
assert checksum==EXPECTED_ZIP_SHA256, 'ZIP không khớp gói đã kiểm tra'
ROOT.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(DATASET_ZIP) as z:
    assert len(z.namelist())==len(set(z.namelist()))
    for info in z.infolist():
        rel=Path(info.filename)
        assert not rel.is_absolute() and '..' not in rel.parts
        assert (ROOT/rel).resolve().is_relative_to(ROOT.resolve())
    z.extractall(ROOT)
sys.path.insert(0,str(ROOT))
from verify_dataset import verify
from colab_checkpoint_support import DriveCheckpointStore,TrainingMonitor,file_sha256,atomic_copy,atomic_json
manifest=verify(ROOT)
DATA=ROOT/'data-colab.yaml'
config=yaml.safe_load((ROOT/'data.yaml').read_text())
config['path']=str(ROOT)
DATA.write_text(yaml.safe_dump(config,sort_keys=False),encoding='utf-8')
if not FULL_TRAIN:
    rng=random.Random(42)
    for split,count in [('train',200),('validation',80)]:
        members=[r for r in manifest['records'] if r['split']==split]
        rng.shuffle(members)
        (ROOT/f'smoke-{split}.txt').write_text('\n'.join(str(ROOT/r['image']) for r in members[:count])+'\n',encoding='utf-8')
    config.update(train=str(ROOT/'smoke-train.txt'),val=str(ROOT/'smoke-validation.txt'))
    DATA=ROOT/'data-smoke.yaml'
    DATA.write_text(yaml.safe_dump(config,sort_keys=False),encoding='utf-8')
print('Training YAML:',DATA)


In [ ]:
from ultralytics import settings
settings.update({'tensorboard':True})
%load_ext tensorboard
%tensorboard --logdir /content/anpr-pose-runs


In [ ]:
from ultralytics import YOLO,__version__
from IPython.display import display,HTML
assert __version__=='8.4.183'
contract=dict(manifest_sha256=file_sha256(ROOT/'manifest.json'),full_train=FULL_TRAIN,
              epochs=EPOCHS,imgsz=IMGSZ,kpt_shape=[4,3],ultralytics=__version__,
              initial_weights_sha256=manifest['initial_weights_sha256'],seed=42)
store=DriveCheckpointStore(DRIVE_RUN,contract,resume=RESUME)
LOCAL_RUN.mkdir(parents=True,exist_ok=True)
completed=False
if RESUME:
    checkpoint,saved=store.restore(LOCAL_RUN,allow_completed=True)
    completed=bool(saved.get('completed')) or saved['epoch']>=EPOCHS
    model=YOLO(str(checkpoint))
    print('Restored epoch:',saved['epoch'])
else:
    model=YOLO(str(ROOT/'weights/yolo11n-pose.pt'))
    assert model.task=='pose' and model.model.yaml['kpt_shape']==[17,3]
shutil.copyfile(ROOT/'manifest.json',LOCAL_RUN/'dataset-manifest.json')
with (LOCAL_RUN/'environment-freeze.txt').open('w') as f:
    subprocess.run([sys.executable,'-m','pip','freeze'],stdout=f,check=True)
atomic_copy(LOCAL_RUN/'dataset-manifest.json',DRIVE_RUN/'dataset-manifest.json')
atomic_copy(LOCAL_RUN/'environment-freeze.txt',DRIVE_RUN/'environment-freeze.txt')
panel=display(HTML('Chuẩn bị train...'),display_id=True)
monitor=TrainingMonitor(store,lambda markup:panel.update(HTML(markup)))
for event,callback in {'on_train_start':monitor.start,'on_train_epoch_start':monitor.epoch_start,
                       'on_train_batch_end':monitor.batch_end,'on_val_start':monitor.val_start,
                       'on_val_batch_end':monitor.val_batch_end,'on_model_save':monitor.saved,
                       'on_train_end':monitor.finished}.items():
    model.add_callback(event,callback)
if not completed:
    try:
        if RESUME:
            model.train(resume=True,data=str(DATA),device=0,batch=BATCH,workers=2,save_dir=str(LOCAL_RUN))
        else:
            model.train(data=str(DATA),epochs=EPOCHS,imgsz=IMGSZ,batch=BATCH,device=0,
                        workers=2,seed=42,deterministic=True,cache=False,
                        fliplr=0.0,flipud=0.0,mosaic=0.0,mixup=0.0,cutmix=0.0,
                        degrees=0.0,shear=0.0,perspective=0.0,translate=0.05,scale=0.2,
                        patience=20,project=str(LOCAL_RUN.parent),name=RUN_ID,
                        exist_ok=True,save=True,save_period=-1,plots=True)
    except BaseException:
        monitor.phase='interrupted'
        monitor.update(True)
        raise
else:
    print('Run đã hoàn tất; không train lại.')
print('RUN_ID:',RUN_ID,'| checkpoint:',store.state())


In [ ]:
if not FULL_TRAIN:
    print('Chạy thử hoàn tất. Chọn FULL_TRAIN=True, RESUME=False, RUN_ID="" rồi chạy lại từ ô cấu hình.')
else:
    best=LOCAL_RUN/'weights/best.pt'
    assert best.is_file()
    trained=YOLO(str(best))
    assert trained.model.yaml['kpt_shape']==[4,3], 'Model phải có đúng bốn điểm'
    exports=DRIVE_RUN/'exports';exports.mkdir(exist_ok=True)
    atomic_copy(best,exports/'best.pt')
    atomic_copy(ROOT/'manifest.json',exports/'dataset-manifest.json')
    print('Đã lưu best.pt:',exports)
    # Tùy chọn export ONNX sau khi kiểm tra chất lượng validation.
    EXPORT_ONNX=False
    if EXPORT_ONNX:
        onnx_path=Path(trained.export(format='onnx',imgsz=IMGSZ,opset=17,dynamic=False,simplify=True,device='cpu'))
        atomic_copy(onnx_path,exports/'best.onnx')
